# What does the player-share model predict?

Pick one Six Nations round and inspect its forecasts. The notebook fits on all
matches before that round, then compares player shares with and without
opposition against the recency-weighted mean. The squad comparison uses prices
when the prepared roster is available.

In [1]:
from math import log
from pathlib import Path

import pandas as pd
from IPython.display import display

from six_nations_fantasy.data import load_dataset
from six_nations_fantasy.data.fantasy_fixtures import fixture_path, tournament_rounds
from six_nations_fantasy.data.load import DATA_DIR
from six_nations_fantasy.evaluation.pipeline import fit_predict, recorded_lineup
from six_nations_fantasy.evaluation.selection import prediction_lineup, scored_player_fixtures
from six_nations_fantasy.evaluation.squads import select_squad, squad_solver
from six_nations_fantasy.models import PlayerShareModel, RecencyWeightedMean
from six_nations_fantasy.models.player_share import component_units
from six_nations_fantasy.score import simple_score

data_dir = DATA_DIR   # Or Path("/path/to/your/data")
year, round_number = 2026, 1
tau = 0.25
stat = "tries"        # Choose a row from the component summary below.
team = None          # Show every side, or set a name such as "France".

In [2]:
dataset = load_dataset(data_dir)
schedule = dict(tournament_rounds(dataset.matches, year))[round_number]
round_start = schedule["date"].min()
training = dataset.matches_before(round_start)
held = dataset.select_matches(schedule["match_id"])

roster_file = fixture_path(data_dir, year)
roster = pd.read_parquet(roster_file) if roster_file.exists() else None
if roster is not None and not roster["round"].eq(round_number).any():
    roster = None
scored = scored_player_fixtures(held, round_number, roster)
fixtures = prediction_lineup(scored)
print(f"Training cutoff {round_start.date()} | {len(training.matches):,} matches")
print(f"{len(fixtures)} players from {'announced rosters' if roster is not None else 'recorded team sheets'}")

scored = scored.copy()
if "name" not in scored:
    registry_file = data_dir / "players.parquet"
    if registry_file.exists():
        registry = pd.read_parquet(registry_file)
        if "player_id" in registry:
            registry = registry.set_index("player_id")
        scored["name"] = scored["player_id"].map(registry["name"])
    else:
        scored["name"] = scored["player_id"].astype(str)

Training cutoff 2026-02-05 | 1,828 matches
138 players from announced rosters


In [3]:
share = PlayerShareModel(tau=tau)
without_opposition = PlayerShareModel(tau=0)
recency = RecencyWeightedMean(scorer=simple_score, half_life_days=7 * log(0.5) / log(0.85))

comparison = {
    "share": fit_predict(share, training, fixtures),
    "share_no_opposition": fit_predict(without_opposition, training, fixtures),
    "recency": fit_predict(recency, training, fixtures),
}
forecasts = pd.DataFrame(comparison).assign(actual=scored["actual"])

/home/julian/code/fantasy-rugby/six_nations_fantasy/models/player_share.py:83: UserWarning: Excluded from training: 1 incomplete team sheets (expected shirts 1–23), first: {'match_id': 276171, 'team': 'Bulls'}
  rows, team_rows, teams = full_lineups(frame, training=True)
/home/julian/code/fantasy-rugby/six_nations_fantasy/models/player_share.py:83: UserWarning: Excluded from training: 1 incomplete team sheets (expected shirts 1–23), first: {'match_id': 276171, 'team': 'Bulls'}
  rows, team_rows, teams = full_lineups(frame, training=True)


## Team multipliers across the scoring stats

For each stat, the team forecast is the historical starting average times the
competition, home/away, team and opponent multipliers. The table shows how big
the fitted opponent multipliers are in this round. It also compares the two
**refitted** team forecasts. Their difference can include changes in the team
and competition coefficients when the opponent term is removed.

In [4]:
keys = ["match_id", "team"]
recorded = recorded_lineup(held)
actual_counts = component_units(held.performances)
actual_team = actual_counts.groupby([recorded[key].to_numpy() for key in keys]).sum()
actual_team.index.names = keys

team_tables = {}
for component in share.opponent_scales:
    with_opp, _ = share.explain_component(fixtures, component)
    without_opp, _ = without_opposition.explain_component(fixtures, component)
    team_tables[component] = (
        with_opp.merge(
            without_opp[keys + ["expected_units"]].rename(
                columns={"expected_units": "expected_without_opposition"}
            ),
            on=keys, validate="one_to_one",
        )
        .join(actual_team[component].rename("observed_units"), on=keys)
        .assign(change_with_opposition=lambda rows:
                rows["expected_units"] - rows["expected_without_opposition"])
    )

summary = {}
for component, sides in team_tables.items():
    with_error = sides["expected_units"] - sides["observed_units"]
    without_error = sides["expected_without_opposition"] - sides["observed_units"]
    summary[component] = {
        "smallest opponent multiplier": sides["opponent_multiplier"].min(),
        "largest opponent multiplier": sides["opponent_multiplier"].max(),
        "mean absolute change in team units": sides["change_with_opposition"].abs().mean(),
        "team MAE with opposition": with_error.abs().mean(),
        "team MAE without opposition": without_error.abs().mean(),
    }
display(pd.DataFrame.from_dict(summary, orient="index").round(3))
display(team_tables[stat].drop(columns=["date", "competition"]).round(3))

,smallest opponent multiplier,largest opponent multiplier,mean absolute change in team units,team MAE with opposition,team MAE without opposition
tries,0.703,1.482,0.562,1.164,0.840
try_assists,0.643,1.505,0.517,1.321,1.040
conversions,0.734,1.377,0.301,0.885,0.678
penalty_goals,0.711,1.315,0.237,0.881,0.727
drop_goals,0.977,1.041,0.001,0.021,0.021
defenders_beaten,0.769,1.231,2.022,6.643,7.089
carried_meters,0.860,1.155,34.745,140.121,112.120
completed_tackles,0.872,1.130,12.929,23.118,29.352
turnovers_won,0.877,1.145,0.671,2.304,1.884
offloads,0.858,1.193,0.558,3.879,3.641


,match_id,team,opponent,home,competition_multiplier,home_multiplier,team_multiplier,opponent_multiplier,baseline_units,expected_units,expected_without_opposition,observed_units,change_with_opposition
0,290465,France,Ireland,True,0.853,1.195,1.514,0.703,3.185,3.453,5.052,5,-1.599
1,290465,Ireland,France,False,0.853,0.826,1.403,0.882,3.185,2.780,3.368,2,-0.588
2,290466,Italy,Scotland,True,0.853,1.195,0.657,0.990,3.185,2.111,2.023,2,0.088
3,290466,Scotland,Italy,False,0.853,0.826,1.007,1.482,3.185,3.351,2.344,2,1.007
4,290467,England,Wales,True,0.853,1.195,0.983,1.079,3.185,3.444,3.370,6,0.073
5,290467,Wales,England,False,0.853,0.826,0.717,1.018,3.185,1.639,1.622,1,0.017


## How the team total is split

Multiply the position weight by the player multiplier, then divide by the total
weight on that sheet. The resulting share is the fraction of the team's units
assigned to the player. Observed shares use the recorded team total. A team with
zero observed units has no observed shares. Metres in these tables are raw
metres, with negative values counted as zero.

In [5]:
_, allocation = share.explain_component(fixtures, stat)
actual_players = recorded[["match_id", "player_id"]].assign(
    observed_units=actual_counts[stat].to_numpy()
)
allocation = (
    allocation.assign(name=scored["name"].to_numpy())
    .merge(actual_players, on=["match_id", "player_id"], how="left", validate="one_to_one")
    .join(actual_team[stat].rename("observed_team_units"), on=keys)
)
allocation["observed_share"] = (
    allocation["observed_units"].fillna(0)
    / allocation["observed_team_units"].where(allocation["observed_team_units"].ne(0))
)
columns = [
    "team", "name", "number", "position_role", "position_weight", "player_multiplier",
    "share", "observed_share", "expected_units", "observed_units",
]
selected = allocation if team is None else allocation.loc[allocation["team"].eq(team)]
display(selected.sort_values(["team", "share"], ascending=[True, False])[columns].round(3))

,team,name,number,position_role,position_weight,player_multiplier,share,observed_share,expected_units,observed_units
105,England,Immanuel Feyi-Waboso,14,Back Three,2.158,2.092,0.119,0.000,0.409,NaN
102,England,Henry Arundell,11,Back Three,2.158,2.038,0.116,0.500,0.399,3.0
104,England,Tommy Freeman,13,Centre,1.315,2.640,0.091,0.167,0.315,1.0
93,England,Jamie George,2,Hooker,1.245,2.171,0.071,0.000,0.245,0.0
99,England,Ben Earl,8,Back-row,1.154,2.130,0.065,0.167,0.223,1.0
...,...,...,...,...,...,...,...,...,...,...
118,Wales,Dafydd Jenkins,4,Second-row,0.633,0.770,0.014,0.000,0.023,0.0
133,Wales,Ben Carter,19,Substitute,0.479,0.973,0.014,0.000,0.022,0.0
132,Wales,Tomas Francis,18,Substitute,0.479,0.966,0.013,0.000,0.022,0.0
130,Wales,Liam Belcher,16,Substitute,0.479,0.938,0.013,0.000,0.021,0.0


## Does the model pick different players?

The first table compares player errors and rank correlation on this round. The
next shows where the share model and recency mean disagree most. When prices are
available, solve the same fantasy squad problem for each forecast and compare
the players selected, captain, supersub and points earned.

In [6]:
model_names = list(comparison)
errors = forecasts[model_names].sub(forecasts["actual"], axis=0)
ranks = forecasts[model_names].rank()
display(pd.DataFrame({
    "player RMSE": errors.pow(2).mean().pow(0.5),
    "rank correlation with recency": ranks.corr()["recency"],
}).round(3))

details = scored[["match_id", "team", "player_id", "name"]].join(forecasts)
details["effect of refitting with opposition"] = details["share"] - details["share_no_opposition"]
details["share minus recency"] = details["share"] - details["recency"]
order = details["share minus recency"].abs().sort_values(ascending=False).index
display(details.loc[order].head(15).round(2))

,player RMSE,rank correlation with recency
share,10.164,0.758
share_no_opposition,9.893,0.762
recency,11.513,1.000


,match_id,team,player_id,name,share,share_no_opposition,recency,actual,effect of refitting with opposition,share minus recency
113,290467,England,111662,Ben Spencer,4.47,4.34,39.00,3.0,0.12,-34.53
114,290467,England,189326,Marcus Smith,8.00,7.86,38.29,9.0,0.15,-30.28
9,290465,France,190344,Matthieu Jalibert,25.25,28.21,48.62,55.0,-2.96,-23.36
104,290467,England,156918,Tommy Freeman,23.82,23.31,43.20,55.0,0.51,-19.37
45,290465,Ireland,100659,Jack Crowley,5.27,5.78,22.57,6.0,-0.52,-17.31
112,290467,England,239634,Henry Pollock,12.68,12.37,29.28,3.0,0.32,-16.60
20,290465,France,244170,Lenni Nouchi,9.84,10.37,25.78,16.0,-0.53,-15.94
95,290467,England,173801,Alex Coles,17.40,16.61,31.95,11.0,0.79,-14.54
79,290466,Scotland,195298,Jamie Dobie,22.90,19.78,37.11,12.0,3.12,-14.20
125,290467,Wales,137453,Josh Adams,15.28,15.56,28.56,25.0,-0.28,-13.28


In [7]:
if roster is not None:
    solver = squad_solver()
    candidates = scored.join(forecasts[model_names]).dropna(subset=["cost"])
    squads = {name: select_squad(candidates, name, solver) for name in [*model_names, "actual"]}
    recency_ids = set(squads["recency"]["player_id"])
    optimum = squads["actual"]["actual_total"].sum()
    rows = []
    for name, squad in squads.items():
        actual = squad["actual_total"].sum()
        rows.append({
            "selection": "hindsight optimum" if name == "actual" else name,
            "players also picked by recency": len(recency_ids & set(squad["player_id"])),
            "captain": squad.loc[squad["captain"], "name"].iloc[0],
            "supersub": squad.loc[squad["supersub"], "name"].iloc[0],
            "actual points": actual,
            "% of optimum": 100 * actual / optimum if optimum else float("nan"),
        })
    display(pd.DataFrame(rows).round(2))
else:
    print("Prepare this round's fantasy roster to compare squad selections.")

,selection,players also picked by recency,captain,supersub,actual points,% of optimum
0,share,3,Antoine Dupont,Henry Pollock,416.0,56.99
1,share_no_opposition,4,Antoine Dupont,Henry Pollock,370.0,50.68
2,recency,16,Matthieu Jalibert,Ben Spencer,455.0,62.33
3,hindsight optimum,5,Henry Arundell,Nick Timoney,730.0,100.00
